In [17]:
import json
import re
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data/cicids")
STIX_FILE = Path("../data/attck/enterprise-attack.json")
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / "cicids_processed.csv"

In [18]:
#Load ATT&CK STIX Bundle
with open(STIX_FILE, "r", encoding="utf-8") as f:
    bundle = json.load(f)

techniques = {}
for obj in bundle.get("objects", []):
    if obj.get("type") != "attack-pattern" or obj.get("revoked", False):
        continue
    tech_id = None
    for ref in obj.get("external_references", []):
        if ref.get("source_name") == "mitre-attack":
            tech_id = ref.get("external_id")
            break
    if not tech_id:
        continue
    tactics = []
    for phase in obj.get("kill_chain_phases", []):
        if phase.get("kill_chain_name") == "mitre-attack":
            tactics.append(phase["phase_name"].replace("-", " ").title())
    techniques[tech_id] = {
        "name": obj.get("name", "Unknown"),
        "tactic": tactics[0] if tactics else "Unknown"
    }
print(f"Loaded {len(techniques)} ATT&CK techniques")

Loaded 703 ATT&CK techniques


In [19]:
#Label Mapping & Alert Text Builder
raw_map = {
    "FTP-Patator": "T1110.001", "FTP Patator": "T1110.001",
    "SSH-Patator": "T1110.001", "SSH Patator": "T1110.001",
    "DoS slowloris": "T1499.001", "DoS Slowhttptest": "T1499.001",
    "DoS Hulk": "T1499.001", "DoS GoldenEye": "T1499.001",
    "Heartbleed": "T1190", "Web Attack Brute Force": "T1110.001",
    "Web Attack XSS": "T1059.007", "Web Attack Sql Injection": "T1190",
    "Infiltration": "T1105", "Bot": "T1071.001",
    "PortScan": "T1046", "DDoS": "T1498.001", "BENIGN": None,
}

label_map = {}
for label, tech_id in raw_map.items():
    if tech_id is None:
        label_map[label] = {"technique_id": "BENIGN", "technique_name": "Benign Traffic", "tactic": "Benign"}
    else:
        info = techniques.get(tech_id, {"name": "Unknown", "tactic": "Unknown"})
        label_map[label] = {"technique_id": tech_id, "technique_name": info["name"], "tactic": info["tactic"]}

def normalise_label(label):
    if not isinstance(label, str):
        return "BENIGN"
    label = re.sub(r"[^\x00-\x7F]+", " ", label)
    label = re.sub(r"\s*[-–—]\s*", " ", label)
    return re.sub(r"\s+", " ", label).strip()

def build_alert_text(row):
    dst_port = int(float(row.get("Destination Port", 0)))
    duration = int(float(row.get("Flow Duration", 0)))
    fwd_pkts = int(float(row.get("Total Fwd Packets", 0)))
    bwd_pkts = int(float(row.get("Total Backward Packets", 0)))
    flow_bps = round(float(row.get("Flow Bytes/s", 0.0)), 2)
    syn_flag = int(float(row.get("SYN Flag Count", 0)))
    ack_flag = int(float(row.get("ACK Flag Count", 0)))
    
    port_map = {21:"FTP service", 22:"SSH service", 53:"DNS service", 80:"HTTP service", 443:"HTTPS service", 8080:"HTTP-alt service"}
    service = port_map.get(dst_port, f"unknown service on port {dst_port}")
    
    behavior = []
    if duration < 100: behavior.append("very short connection")
    elif duration > 1_000_000: behavior.append("long duration")
    total_pkts = fwd_pkts + bwd_pkts
    if total_pkts <= 2: behavior.append("low packet count")
    elif total_pkts > 1000: behavior.append("high packet count")
    if flow_bps > 1_000_000: behavior.append("high volume traffic")
    behavior_text = ". ".join(behavior) if behavior else "observed traffic pattern"
    
    flag_text = ", ".join([f for f, c in [("SYN", syn_flag), ("ACK", ack_flag)] if c > 0]) or "none"
    return f"Network flow to {service}. {behavior_text}. Duration {duration} microseconds. Forward packets {fwd_pkts}. Backward packets {bwd_pkts}. Flow bytes per second {flow_bps}. TCP flags: {flag_text}."

In [20]:
#Load, Process & Save
all_frames = []
csv_files = sorted(DATA_DIR.glob("*.csv"))

for filepath in csv_files:
    df = pd.read_csv(filepath, low_memory=False, encoding="latin-1")
    df.columns = df.columns.str.strip()
    df["Label"] = df["Label"].apply(normalise_label)
    
    attacks = df[df["Label"] != "BENIGN"].copy()
    benign = df[df["Label"] == "BENIGN"].sample(frac=0.10, random_state=42).copy()
    df = pd.concat([attacks, benign], ignore_index=True)
    
    df["attck_technique_id"] = df["Label"].map(lambda x: label_map.get(x, {}).get("technique_id", "UNMAPPED"))
    df["attck_technique_name"] = df["Label"].map(lambda x: label_map.get(x, {}).get("technique_name", "Unknown"))
    df["attck_tactic"] = df["Label"].map(lambda x: label_map.get(x, {}).get("tactic", "Unknown"))
    df["alert_text"] = df.apply(build_alert_text, axis=1)
    
    keep_cols = ["Destination Port", "Flow Duration", "Total Fwd Packets", "Total Backward Packets", 
                 "Flow Bytes/s", "SYN Flag Count", "RST Flag Count", "ACK Flag Count", "Label", 
                 "attck_technique_id", "attck_technique_name", "attck_tactic", "alert_text"]
    all_frames.append(df[[c for c in keep_cols if c in df.columns]])

combined = pd.concat(all_frames, ignore_index=True)
combined.index.name = "alert_id"
combined.to_csv(OUTPUT_FILE)
print(f"Saved {len(combined)} rows to {OUTPUT_FILE}")

Saved 784957 rows to ../data/processed/cicids_processed.csv
